In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO Y VALIDACIÓN CRUZADA
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota-v2"
mlflow.set_experiment(experiment_name)

N_SPLITS = 5
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Mantener exactamente el mismo subconjunto supervisado que en los modelos de árboles
df_supervised = df[df["v_ref_2026"].notnull()].copy().reset_index(drop=True)

X_base = df_supervised[["v_ref_2025"]].copy()
y_sup = df_supervised["v_ref_2026"].copy()


def log_cv_metrics(metricas_dict: dict) -> None:
    """Calcula media, desviación estándar y brecha de sobreajuste, y las registra en MLflow."""
    resumen = {
        "cv_r2_train_mean": float(np.mean(metricas_dict["r2_train"])),
        "cv_r2_train_std": float(np.std(metricas_dict["r2_train"])),
        "cv_r2_val_mean": float(np.mean(metricas_dict["r2_val"])),
        "cv_r2_val_std": float(np.std(metricas_dict["r2_val"])),
        "cv_r2_gap_mean": float(
            np.mean(np.array(metricas_dict["r2_train"]) - np.array(metricas_dict["r2_val"]))
        ),
        "cv_mae_train_mean": float(np.mean(metricas_dict["mae_train"])),
        "cv_mae_val_mean": float(np.mean(metricas_dict["mae_val"])),
        "cv_mae_val_std": float(np.std(metricas_dict["mae_val"])),
        "cv_rmse_val_mean": float(np.mean(metricas_dict["rmse_val"])),
        "cv_rmse_val_std": float(np.std(metricas_dict["rmse_val"])),
    }
    mlflow.log_metrics(resumen)
    for k, v in resumen.items():
        print(f"  {k}: {v:,.4f}")


class ModeloIngenuoMediana(mlflow.pyfunc.PythonModel):
    """Modelo baseline que proyecta 2026 multiplicando v_ref_2025 por el crecimiento mediano."""

    def __init__(self, factor_crecimiento: float):
        self.factor_crecimiento = factor_crecimiento

    def predict(self, context, model_input: pd.DataFrame) -> np.ndarray:
        return model_input["v_ref_2025"].to_numpy() * self.factor_crecimiento


# ==============================================================================
# RUN 1: BASELINE INGENUO (Crecimiento Mediano 2025 -> 2026)
# ==============================================================================
print("\n--- 2. EVALUANDO BASELINE INGENUO (CRECIMIENTO MEDIANO) ---")
metricas_ingenuo = {
    "r2_train": [], "r2_val": [],
    "mae_train": [], "mae_val": [], "rmse_val": []
}
factores_fold = []

for train_idx, val_idx in kf.split(X_base):
    X_tr, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_tr, y_val = y_sup.iloc[train_idx], y_sup.iloc[val_idx]

    # El factor mediano se calcula SOLO con el conjunto de entrenamiento del pliegue
    factor_fold = float((y_tr / X_tr["v_ref_2025"]).median())
    factores_fold.append(factor_fold)

    y_pred_tr = X_tr["v_ref_2025"].to_numpy() * factor_fold
    y_pred_val = X_val["v_ref_2025"].to_numpy() * factor_fold

    metricas_ingenuo["r2_train"].append(r2_score(y_tr, y_pred_tr))
    metricas_ingenuo["r2_val"].append(r2_score(y_val, y_pred_val))
    metricas_ingenuo["mae_train"].append(mean_absolute_error(y_tr, y_pred_tr))
    metricas_ingenuo["mae_val"].append(mean_absolute_error(y_val, y_pred_val))
    metricas_ingenuo["rmse_val"].append(np.sqrt(mean_squared_error(y_val, y_pred_val)))

with mlflow.start_run(run_name="Baseline_Ingenuo_Mediana"):
    factor_global = float((y_sup / X_base["v_ref_2025"]).median())

    mlflow.log_params({
        "modelo": "Baseline_Ingenuo_Mediana",
        "features": "v_ref_2025",
        "target_transform": "Ninguna",
        "cv_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "factor_crecimiento_global": round(factor_global, 6),
        "factor_crecimiento_cv_mean": round(float(np.mean(factores_fold)), 6),
    })

    log_cv_metrics(metricas_ingenuo)

    mlflow.pyfunc.log_model(
        artifact_path="baseline_ingenuo_model",
        python_model=ModeloIngenuoMediana(factor_crecimiento=factor_global),
        input_example=X_base.head(5),
    )


# ==============================================================================
# RUN 2: BASELINE REGRESIÓN LINEAL SIMPLE (v_ref_2026 ~ v_ref_2025)
# ==============================================================================
print("\n--- 3. EVALUANDO BASELINE REGRESIÓN LINEAL SIMPLE ---")
metricas_lr = {
    "r2_train": [], "r2_val": [],
    "mae_train": [], "mae_val": [], "rmse_val": []
}

for train_idx, val_idx in kf.split(X_base):
    X_tr, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_tr, y_val = y_sup.iloc[train_idx], y_sup.iloc[val_idx]

    lr_fold = LinearRegression()
    lr_fold.fit(X_tr, y_tr)

    y_pred_tr = lr_fold.predict(X_tr)
    y_pred_val = lr_fold.predict(X_val)

    metricas_lr["r2_train"].append(r2_score(y_tr, y_pred_tr))
    metricas_lr["r2_val"].append(r2_score(y_val, y_pred_val))
    metricas_lr["mae_train"].append(mean_absolute_error(y_tr, y_pred_tr))
    metricas_lr["mae_val"].append(mean_absolute_error(y_val, y_pred_val))
    metricas_lr["rmse_val"].append(np.sqrt(mean_squared_error(y_val, y_pred_val)))

with mlflow.start_run(run_name="Baseline_Regresion_Lineal_Simple"):
    # Entrenar modelo final con todo el conjunto supervisado para empaquetarlo
    lr_final = LinearRegression()
    lr_final.fit(X_base, y_sup)

    mlflow.log_params({
        "modelo": "LinearRegression",
        "features": "v_ref_2025",
        "target_transform": "Ninguna",
        "cv_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "coef_v_ref_2025": round(float(lr_final.coef_[0]), 6),
        "intercept": round(float(lr_final.intercept_), 2),
    })

    log_cv_metrics(metricas_lr)

    mlflow.sklearn.log_model(
        sk_model=lr_final,
        artifact_path="baseline_lr_model",
        input_example=X_base.head(5),
    )